# The Set $\mathbb{Q}$ is Countable

In [ ]:
import java.util.stream.*;

The function `gcd(a, b)` computes the *greatest common divisor* of `a` and `b`. 

In [ ]:
long gcd(long a, long b) {
    if (b == 0) {
        return a;
    }
    return gcd(b, a % b);
}

In [ ]:
gcd(120, 100)

The Java library does not contain a class for fractions.  Therefore, we define the record class `Fraction`.  A fraction $\frac{n}{d}$ is stored as a pair consisting of the `numerator` $n$ and the `denominator` $d$.  The *compact constructor* `public Fraction { ... }` is executed every time a fraction is created.  It reduces the fraction by dividing both numerator and denominator by their greatest common divisor.  Furthermore, it ensures that the denominator is positive.  Therefore, the numerator determines the sign of the fraction.

  * The method `add` adds two fractions using the formula
    $$ \frac{n_1}{d_1} + \frac{n_2}{d_2} = \frac{n_1 \cdot d_2 + n_2 \cdot d_1}{d_1 \cdot d_2}. $$
  * The method `toString` returns a string of the form `"n/d"`.  If the denominator is $1$, only the numerator is returned.

In [ ]:
record Fraction(long numerator, long denominator) {
    public Fraction {
        long g = gcd(Math.abs(numerator), Math.abs(denominator));
        if (denominator < 0) {
            g = -g;
        }
        numerator   /= g;
        denominator /= g;
    }

    Fraction add(Fraction other) {
        return new Fraction(numerator * other.denominator + other.numerator * denominator,
                            denominator * other.denominator);
    }

    @Override
    public String toString() {
        return denominator == 1 ? String.valueOf(numerator) : numerator + "/" + denominator;
    }
}

In [ ]:
var result = new Fraction(-1, 2).add(new Fraction(1, 3));
result

In [ ]:
result.numerator()

In [ ]:
result.denominator()

The function `generateRationals` returns an infinite *stream* of all non-negative rational numbers.
* First, the rational number $\frac{0}{1}$ is generated.
* Second, all rational numbers $\frac{p}{q}$ are computed that satisfy $p > 0$, $p + q = 2$ and $\texttt{gcd}(p, q) = 1$,
* Third, all rational numbers $\frac{p}{q}$ are computed that satisfy $p > 0$, $p + q = 3$ and $\texttt{gcd}(p, q) = 1$,
* $\vdots$

A stream can be infinite, as its elements are only computed when they are needed.
* `Stream.iterate(2L, sum -> sum + 1)` is the infinite stream $[2, 3, 4, \cdots]$ of the possible values of $p + q$.
* For every value `sum` of $p + q$, the method `flatMap` computes the stream of all fractions $\frac{p}{q}$ such that $p + q = \texttt{sum}$ and $\texttt{gcd}(p, q) = 1$ and appends these streams.
* `Stream.concat` puts the fraction $\frac{0}{1}$ in front of the resulting stream.

In [ ]:
Stream<Fraction> generateRationals() {
    Stream<Fraction> positive =
        Stream.iterate(2L, sum -> sum + 1)
              .flatMap(sum -> LongStream.range(1, sum)
                                        .filter(numerator -> gcd(numerator, sum - numerator) == 1)
                                        .mapToObj(numerator -> new Fraction(numerator, sum - numerator)));
    return Stream.concat(Stream.of(new Fraction(0, 1)), positive);
}

The method `iterator` turns the stream into an object `g` of type
```
Iterator<Fraction>
```
This iterator has a method `next`.  Every time we call `g.next()` we get the next rational number.
As the set of rational numbers is infinite, there will always be a next rational number.

In [ ]:
var g = generateRationals().iterator();

In [ ]:
var v1 = g.next();
v1

In [ ]:
var v2 = g.next();
v2

In [ ]:
g.next()

In [ ]:
g.next()

The function `gen(cnt)` prints the first `cnt` non-negative rational numbers.  The method `limit(cnt)` returns a stream that consists of the first `cnt` elements of the given stream.  `String.format("%5s", frac)` pads the string representation of `frac` with blanks so that it has a width of five characters.

In [ ]:
void gen(int cnt) {
    generateRationals().limit(cnt).forEach(frac -> System.out.println(String.format("%5s", frac)));
}

In [ ]:
gen(100)